In [ ]:
import pandas as pd
import json
import warnings
import numpy as np
warnings.filterwarnings('ignore')

In [ ]:
# =========================================
# PROCESAMIENTO DE DATOS
# =========================================

# 1. Carga de Datos
url = "../estadistica_stop/ESTADISTICA_DELITO.csv"
df = pd.read_csv(url)

# 2. Calcular Totales
totales = df.groupby(['codcom', 'id_semana'], as_index=False)['frecuencia'].sum()
totales['delito'] = 'Total'
dim_tiempo = df[['id_semana', 'semana_detalle', 'fecha']].drop_duplicates()
totales = totales.merge(dim_tiempo, on='id_semana', how='left')
totales = totales[df.columns]
df = pd.concat([df, totales], ignore_index=True)

# 3. Preparación Temporal
df['fecha'] = pd.to_datetime(df['fecha'])
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month
df['semana_numero'] = df['semana_detalle'].apply(lambda x: int(x[7:9]))
df = df.sort_values(['delito', 'codcom', 'id_semana'])

# 4. Variables Base (Actual, Anterior, Delta)
g = df.groupby(['delito', 'codcom'])
df['casos_semana_actual'] = df['frecuencia']
df['casos_semana_anterior'] = g['frecuencia'].shift(1)
df['delta'] = df['casos_semana_actual'] - df['casos_semana_anterior']

# 5. Acumulados
df['acumulado_anual'] = df.groupby(['delito','codcom','año'])['frecuencia'].cumsum()
df['acumulado_total'] = g['frecuencia'].cumsum()

# 5b. Acumulado Año Anterior (Misma Semana)
df_prev_acum = df[['delito','codcom','año','semana_numero','acumulado_anual']].assign(año=lambda x: x['año'] + 1).rename(columns={'acumulado_anual': 'acumulado_anual_anterior'})
df = df.merge(df_prev_acum, on=['delito','codcom','año','semana_numero'], how='left')

# 6. Medias Móviles
df['media_movil_4s'] = g['frecuencia'].rolling(4, min_periods=1).mean().reset_index(level=[0,1], drop=True)
df['media_movil_8s'] = g['frecuencia'].rolling(8, min_periods=1).mean().reset_index(level=[0,1], drop=True)

# 7. Estadísticas Históricas
df['promedio_hist'] = g['frecuencia'].expanding().mean().reset_index(level=[0,1], drop=True)
df['std_hist'] = g['frecuencia'].expanding().std().reset_index(level=[0,1], drop=True)
df['max_hist'] = g['frecuencia'].expanding().max().reset_index(level=[0,1], drop=True)

# 8. Estadísticas Año Anterior
stats_anuales = df.groupby(['delito','codcom','año'], group_keys=False).apply(lambda x: x.assign(
    promedio_hist_anual = x['frecuencia'].expanding().mean(),
    std_hist_anual      = x['frecuencia'].expanding().std(),
    max_hist_anual      = x['frecuencia'].expanding().max(),
    id_semana_ref       = x['id_semana']
))
stats_prev = stats_anuales.assign(año=lambda x: x['año'] + 1)[['delito','codcom','año','id_semana_ref','promedio_hist_anual','std_hist_anual','max_hist_anual']]
df = df.merge(stats_prev, left_on=['delito','codcom','año','id_semana'], right_on=['delito','codcom','año','id_semana_ref'], how='left').drop(columns='id_semana_ref')

# 9. Variaciones y Metricas Avanzadas
df['var_pct_vs_semana_anterior'] = (df['delta'] / df['casos_semana_anterior']) * 100
df['z_score'] = (df['frecuencia'] - df['promedio_hist']) / df['std_hist']
df['z_score_vs_año_anterior'] = (df['frecuencia'] - df['promedio_hist_anual']) / df['std_hist_anual']
df['conclusion_z'] = pd.cut(df['z_score'], bins=[-np.inf, -2, 2, np.inf], labels=['Bajo', 'Normal', 'Alto'])
df['tendencia_corto_plazo'] = np.where(df['delta'] > 0, 'Alza', np.where(df['delta'] < 0, 'Baja', 'Estable'))
df['racha'] = (df['delta'] > 0).astype(int).groupby((df['delta'] <= 0).cumsum()).cumsum()

idx = g['frecuencia'].idxmax()
df['id_semana_max_hist'] = df.loc[idx, 'id_semana'].reindex(df.index).values

df['alerta_aumento_critico'] = (df['z_score'] > 2) & (df['var_pct_vs_semana_anterior'] > 30)
df['alerta_vs_año_anterior'] = (df['z_score_vs_año_anterior'] > 2) & (df['frecuencia'] > df['max_hist_anual'])

df_prev_casos = df[['delito','codcom','año','semana_numero','frecuencia']].assign(año=lambda x: x['año'] + 1).rename(columns={'frecuencia': 'casos_misma_semana_año_anterior'})
df = df.merge(df_prev_casos, on=['delito','codcom','año','semana_numero'], how='left')

# 10. Merge External Data (Localiza & Factores)
localiza = pd.read_excel(r"D:\GitHub\LOCALIZA_DB\Localiza Chile (1).xlsx")
localiza2 = localiza[['Provincia', 'Comuna', 'Región','Codcom', 'Codreg']].drop_duplicates()
df2 = df.merge(localiza2, left_on="codcom", right_on="Codcom")

df2 = df2.sort_values(['Codreg', 'delito', 'Codcom', 'id_semana'])
df2['ranking_comunal_regional'] = df2.groupby(['Codreg', 'delito', 'id_semana'])['frecuencia'].rank(method='dense', ascending=False)
df2['ranking_comunal_regional_semana_anterior'] = df2.groupby(['Codreg', 'delito', 'Codcom'])['ranking_comunal_regional'].shift(1)

clasePoblacion = pd.read_excel(r"C:\Users\limc_\Downloads\Factores Población.xlsx", sheet_name="Clase Población")
factor = pd.read_excel(r"C:\Users\limc_\Downloads\Factores Población.xlsx", sheet_name="Factores")

clasePoblacion2 = clasePoblacion[['Codcom', 'Población', 'Clase Población']]
clasePoblacion2.columns = ['Codcom', 'poblacion_clase', 'clase_poblacion']
factor2 = factor[['Codcom',  'Año',  'Población','Factor Población']]
factor2.columns = ['Codcom',  'año',  'poblacion','facor_poblacion']

df3 = df2.merge(clasePoblacion2).merge(factor2)
del df3["Codcom"]

# =====================================================
# 11. NUEVOS CÁLCULOS (Solitud Usuario - Cards)
# =====================================================

# --- A. Proyecciones y Tasas ---
df3['semana_numero_safe'] = df3['semana_numero'].replace(0, 1)
df3['proyeccion_anual'] = (df3['acumulado_anual'] / df3['semana_numero_safe']) * 52
df3['tasa_semanal'] = (df3['frecuencia'] / df3['poblacion']) * 100000
df3['tasa_proyectada_anual'] = (df3['proyeccion_anual'] / df3['poblacion']) * 100000

# --- B. Agregaciones Regionales y Nacionales ---
# Nacional (suma de agrupacion)
grp_nac = df3.groupby(['delito', 'id_semana'])
df3['tasa_proyectada_nacional'] = grp_nac['proyeccion_anual'].transform('sum') / grp_nac['poblacion'].transform('sum') * 100000
df3['tasa_semanal_nacional'] = grp_nac['frecuencia'].transform('sum') / grp_nac['poblacion'].transform('sum') * 100000

# Regional
grp_reg = df3.groupby(['Codreg', 'delito', 'id_semana'])
df3['tasa_proyectada_regional'] = grp_reg['proyeccion_anual'].transform('sum') / grp_reg['poblacion'].transform('sum') * 100000
df3['tasa_semanal_regional'] = grp_reg['frecuencia'].transform('sum') / grp_reg['poblacion'].transform('sum') * 100000
df3['casos_semana_regional'] = grp_reg['frecuencia'].transform('sum')
df3['aporte_pct_region'] = (df3['frecuencia'] / df3['casos_semana_regional']) * 100

# --- C. Rankings ---
df3['ranking_regional_proy_anual'] = df3.groupby(['Codreg', 'delito', 'id_semana'])['proyeccion_anual'].rank(method='dense', ascending=False)
df3['ranking_nacional_semanal'] = df3.groupby(['delito', 'id_semana'])['frecuencia'].rank(method='dense', ascending=False)
df3['ranking_nacional_proy_anual'] = df3.groupby(['delito', 'id_semana'])['proyeccion_anual'].rank(method='dense', ascending=False)

grp_cluster = df3.groupby(['clase_poblacion', 'delito', 'id_semana'])
df3['ranking_cluster_semanal'] = grp_cluster['frecuencia'].rank(method='dense', ascending=False)
df3['ranking_cluster_proy_anual'] = grp_cluster['proyeccion_anual'].rank(method='dense', ascending=False)

# Shifts de Rankings (Requerimos orden temporal)
df3 = df3.sort_values(['Codreg', 'delito', 'codcom', 'id_semana'])
g_temp = df3.groupby(['delito', 'codcom'])
df3['ranking_regional_proy_anual_anterior'] = g_temp['ranking_regional_proy_anual'].shift(1)
df3['ranking_nacional_semanal_anterior'] = g_temp['ranking_nacional_semanal'].shift(1)
df3['ranking_nacional_proy_anual_anterior'] = g_temp['ranking_nacional_proy_anual'].shift(1)
df3['ranking_cluster_semanal_anterior'] = g_temp['ranking_cluster_semanal'].shift(1)

# --- D. Stats Adicionales ---
df3['proyeccion_mes_actual'] = df3['media_movil_4s'] * 4.33
df3['promedio_diario_semanal'] = df3['frecuencia'] / 7
df3['promedio_diario_historico'] = df3['promedio_hist'] / 7
total_semanal_comuna = df3.groupby(['codcom', 'id_semana'])['frecuencia'].transform('sum')
df3['share_delito_semanal'] = (df3['frecuencia'] / total_semanal_comuna) * 100

df3.drop(columns=['semana_numero_safe'], inplace=True, errors='ignore')

print("DataFrame Final Listo. Evaluando Columnas...")
print(df3.columns)

In [ ]:
# =========================================
# GUARDADO DE DATOS
# =========================================

# Guardar en JSON comprimido (GZIP) con todas las columnas calculadas
df3.to_json('data3.json.gz', orient='records', compression='gzip', indent=None)

print("Archivo data3.json.gz guardado exitosamente.")